### MENGGABUNGKAN DUA HALAMAN DATA EXCEL
**Tujuan:**
Menggabungkan data dari dua sheet Excel Online Retail II menjadi satu file CSV dengan memastikan tidak ada data yang tercatat lebih dari sekali.
- Input: data/raw/online_retail_II.xlsx
- Output: data/raw/online_retail_II_gabungan.csv

In [ ]:
import pandas as pd
from pathlib import Path

DATA_RAW = Path("..") / "data" / "raw"
PATH_EXCEL = DATA_RAW / "online_retail_II.xlsx"
PATH_CSV = DATA_RAW / "online_retail_II_gabungan.csv"
JUMLAH_BARIS_ASLI = 1_067_371
print("Excel ada:", PATH_EXCEL.exists())

**1. Baca kedua lembar**

Pembacaan data membutuhkan waktu sekitar 2 menit karena data terdiri dari banyak row (1jt+ row). Invoice dan StockCode dibaca sebagai teks karena memuat angka dan huruf 

In [ ]:
lembar = pd.read_excel(
    PATH_EXCEL,
    sheet_name=None,
    dtype={"Invoice": "string", "StockCode": "string"},
)
for nama, df in lembar.items():
    print(nama, df.shape)
    
s1 = lembar["Year 2009-2010"]
s2 = lembar["Year 2010-2011"]
assert len(s1) + len(s2) == JUMLAH_BARIS_ASLI, "Jumlah baris tidak sesuai dataset asli"
print("Total baris:", f"{len(s1) + len(s2):,}")

**2. Memeriksa Baris Sama yang Muncul di Kedua Sheet**

Sheet kedua dimulai pada 1 Desember 2010, sedangkan sheet pertama masih memiliki data hingga 9 Desember 2010. Artinya, ada periode tanggal yang sama pada kedua sheet. Jika langsung digabungkan, transaksi pada periode tersebut berisiko tercatat dua kali (duplikat).

In [ ]:
print("Lembar 1:", s1["InvoiceDate"].min(), "->", s1["InvoiceDate"].max())
print("Lembar 2:", s2["InvoiceDate"].min(), "->", s2["InvoiceDate"].max())

tumpang1 = s1[s1["InvoiceDate"] >= s2["InvoiceDate"].min()]
tumpang2 = s2[s2["InvoiceDate"] <= s1["InvoiceDate"].max()]
print("Baris tumpang tindih di lembar 1:", len(tumpang1))
print("Baris tumpang tindih di lembar 2:", len(tumpang2))

kolom = list(tumpang1.columns)
a = tumpang1.sort_values(kolom).reset_index(drop=True)
b = tumpang2.sort_values(kolom).reset_index(drop=True)
identik = a.equals(b)
print("Isi tumpang tindih identik:", identik)
assert identik, "Tumpang tindih tidak identik, jangan buang baris sebelum diperiksa"


**Metode Pembersihan Data I:**
Data pada sheet pertama yang memiliki periode tanggal sama dengan sheet kedua dihapus karena datanya identik. Penghapusan dilakukan berdasarkan tanggal agar data transaksi yang memang tercatat lebih dari sekali dalam satu sheet tidak ikut terhapus.

**3. Gabungkan tanpa tumpang tindih dan simpan**

In [ ]:
gabungan = pd.concat(
    [s1[s1["InvoiceDate"] < s2["InvoiceDate"].min()], s2],
    ignore_index=True,
)

dibuang = len(s1) + len(s2) - len(gabungan)
print("Bentuk data gabungan:", gabungan.shape)
print("Baris yang dibuang:", f"{dibuang:,}")
assert dibuang == len(tumpang1), "Jumlah baris dibuang harus sama dengan baris tumpang tindih"

gabungan.to_csv(PATH_CSV, index=False)
print("Tersimpan di:", PATH_CSV.resolve())

**4. Periksa hasil simpanan**

In [ ]:
cek = pd.read_csv(
    PATH_CSV,
    dtype={"Invoice": "string", "StockCode": "string"},
    parse_dates=["InvoiceDate"],
)
print(cek.shape)
cek.head()